# Project 11 -- Ursula Leverette

**TA Help:** 
   
 

**Collaboration:** My Friend in CS, My Uncle, Another Student, etc., list names of any other people who helped you

(describe the tasks that they helped you with)
- For example: helped figuring out how to load the dataset.
- Another example: helped debug error with my plot.

**Internet Resources:** MongoDB.com

reading for additional understaning and context 
Details and examples for flattening arrays in aggregation pipelines.
https://youtu.be/GL7V1TCPG1U?si=tgGbpiYZ-W2grHJD

https://www.mongodb.com/docs/manual/reference/operator/aggregation/unwind/
https://youtu.be/MYTpEjf8JIM?si=hNEZsFhMqiPj1HYC

Syntax and usage for grouping and accumulator expressions.
https://www.mongodb.com/docs/manual/reference/operator/aggregation/group/
https://youtu.be/xjaE20kM1lw?si=vGGnBwZ4WiQ3PpZz

Comprehensive guide to all aggregation pipeline stages, operators, and examples.
https://www.mongodb.com/docs/manual/aggregation/

**ChatGPT, Gemini, Claude, etc:** Any language models or generative AI chatbots that helped you.

(if you used any such tools, please tell us here)
- For example:  I asked ChatGPT how to define a new data frames
- Another example:  Gemini told me how to make a function for sorting my data

**OVERALL MESSAGE:** Any time that you used anything except your brain to solve the questions in these projects, you need to disclose such resources at the start of the project, with details about your usage of the tools.

**YOUR OWN WORK:** Even when you utilize other resources, do NOT just copy and paste.  Write all explanations in your own words, using several sentences in English, which are understandable and which you wrote (and did not just copy and paste).

## Question 1
 
- 1.1. Create a database called 'ecommerce_db' with collections for 'products', 'customers', and 'orders'.
- 1.2. Load the comprehensive e-commerce dataset from the JSON file into the appropriate collections.
- 1.3. Write a query to find all products in the "Electronics" category and count how many there are.
- 1.5. Write a query to find all customers from California (state: "CA") and count how many there are.
- 1.7. Write a query to find all orders with status "completed" and count how many there are.
- 1.9. Explain how dot notation works for querying nested fields in MongoDB (like address.state or specifications.brand) - I think of dot notation as a way to drill down into sub objects or arrays inside a document, just like referencing a column in a table, but with multiple levels.

In [1]:
from pymongo import MongoClient
import pandas as pd
import json

# create a client to connect to the local MongoDB instance
client = MongoClient('mongodb://localhost:27017/')

# Create or connect to a database
db = client['ecommerce_db']

# Create or connect to collections
products = db['products']
customers = db['customers']
orders = db['orders']

In [2]:
client.drop_database(db.name)

In [3]:
# Load the e-commerce dataset from JSON file
with open('/anvil/projects/tdm/data/ecommerce/fake_ecommerce_dataset.json', 'r') as file:
    dataset = json.load(file)

# Insert the data into MongoDB
products.insert_many(dataset['products'])
customers.insert_many(dataset['customers'])
orders.insert_many(dataset['orders'])

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [4]:
# look inside one customer page and see what sections it has (like customer_id, name, address) and what kind of info is in each section.
sample_customer = customers.find_one() # Get the structure of a customer object
for key, value in sample_customer.items(): 
    print(f"{key}: {type(value).__name__}\n Value: {value if not isinstance(value, str) 
    or len(str(value)) <= 100 else str(value)[:100]+'...'}\n")
  

_id: ObjectId
 Value: 690fb60084264622b1a9bdfa

customer_id: str
 Value: C001

name: str
 Value: John Doe

email: str
 Value: john@email.com

phone: str
 Value: 555-0101

address: dict
 Value: {'street': '123 Main St', 'city': 'New York', 'state': 'NY', 'zip': '10001', 'country': 'USA'}

preferences: dict
 Value: {'categories': ['Electronics', 'Sports'], 'newsletter': True, 'notifications': True, 'language': 'English'}

created_at: str
 Value: 2024-01-01T00:00:00Z



In [5]:
# look inside one product page and see what sections it has (like product_id, name, price) and what kind of info is in each section.
sample_product = products.find_one() # Get the structure of a product object
for key, value in sample_product.items(): 
    print(f"{key}: {type(value).__name__}\n Value: {value if not isinstance(value, str) 
    or len(str(value)) <= 100 else str(value)[:100]+'...'}\n")

_id: ObjectId
 Value: 690fb5ff84264622b1a9bdeb

product_id: str
 Value: P001

name: str
 Value: Wireless Headphones

category: str
 Value: Electronics

price: float
 Value: 99.99

stock: int
 Value: 50

specifications: dict
 Value: {'brand': 'TechSound', 'color': 'Black', 'battery_life': '20 hours', 'connectivity': 'Bluetooth 5.0', 'weight': '250g', 'warranty': '2 years'}

reviews: list
 Value: [{'rating': 5, 'comment': 'Great sound quality!', 'user': 'john_doe', 'date': '2024-01-15'}, {'rating': 4, 'comment': 'Good value for money', 'user': 'jane_smith', 'date': '2024-01-20'}, {'rating': 5, 'comment': 'Excellent noise cancellation', 'user': 'audio_fan', 'date': '2024-02-01'}]

tags: list
 Value: ['audio', 'wireless', 'bluetooth', 'headphones']

created_at: str
 Value: 2024-01-01T00:00:00Z



In [6]:
# look inside one orders page and see what sections it has (like order_date, status, total_amount) and what kind of info is in each section.
sample_order = orders.find_one() # Get the structure of a orders object
for key, value in sample_order.items(): 
    print(f"{key}: {type(value).__name__}\n Value: {value if not isinstance(value, str) 
    or len(str(value)) <= 100 else str(value)[:100]+'...'}\n")

_id: ObjectId
 Value: 690fb60084264622b1a9be09

order_id: str
 Value: O001

customer_id: str
 Value: C001

order_date: str
 Value: 2024-01-15T10:30:00Z

status: str
 Value: completed

total_amount: float
 Value: 229.98

shipping_address: dict
 Value: {'street': '123 Main St', 'city': 'New York', 'state': 'NY', 'zip': '10001', 'country': 'USA'}

items: list
 Value: [{'product_id': 'P001', 'quantity': 1, 'price': 99.99, 'subtotal': 99.99}, {'product_id': 'P003', 'quantity': 1, 'price': 129.99, 'subtotal': 129.99}]

payment_method: str
 Value: credit_card

shipping_cost: float
 Value: 9.99

tax: float
 Value: 18.0



In [7]:
# Examine a sample product document to understand its structure

sample_product = products.find_one()
print("Sample Product Structure:")
print(f"Product Name: {sample_product['name']}")
print(f"Category: {sample_product['category']}")
print(f"Price: ${sample_product['price']}")
print(f"Specifications: {sample_product['specifications']}")
print(f"Number of Reviews: {len(sample_product['reviews'])}")
print(f"Tags: {sample_product['tags']}")

Sample Product Structure:
Product Name: Wireless Headphones
Category: Electronics
Price: $99.99
Specifications: {'brand': 'TechSound', 'color': 'Black', 'battery_life': '20 hours', 'connectivity': 'Bluetooth 5.0', 'weight': '250g', 'warranty': '2 years'}
Number of Reviews: 3
Tags: ['audio', 'wireless', 'bluetooth', 'headphones']


In [8]:
# Examine a sample customers document to understand its structure

sample_customer = customers.find_one()
print("Sample Customer Structure:")
print(f"Customer_Name: {sample_customer['name']}")
print(f"Customer_Email: {sample_customer['email']}")
print(f"Customer_Phone: ${sample_customer['phone']}")
print(f"Address: {sample_customer['address']}")
print(f"Preferences: {sample_customer['preferences']}")


Sample Customer Structure:
Customer_Name: John Doe
Customer_Email: john@email.com
Customer_Phone: $555-0101
Address: {'street': '123 Main St', 'city': 'New York', 'state': 'NY', 'zip': '10001', 'country': 'USA'}
Preferences: {'categories': ['Electronics', 'Sports'], 'newsletter': True, 'notifications': True, 'language': 'English'}


In [9]:
# Examine a sample orders document to understand its structure

sample_order = orders.find_one()
print("Sample Order Structure:")
print(f"Order_ID: {sample_order['order_id']}")
print(f"Order_Date: {sample_order['order_date']}")
print(f"Order_Status: ${sample_order['status']}")
print(f"Total_Order_Amount: {sample_order['total_amount']}")
print(f"Shipping_Address: {sample_order['shipping_address']}")
print(f"Items: {sample_order['items']}")


Sample Order Structure:
Order_ID: O001
Order_Date: 2024-01-15T10:30:00Z
Order_Status: $completed
Total_Order_Amount: 229.98
Shipping_Address: {'street': '123 Main St', 'city': 'New York', 'state': 'NY', 'zip': '10001', 'country': 'USA'}
Items: [{'product_id': 'P001', 'quantity': 1, 'price': 99.99, 'subtotal': 99.99}, {'product_id': 'P003', 'quantity': 1, 'price': 129.99, 'subtotal': 129.99}]


In [10]:
# Find products in a specific category
electronics = list(products.find({"category": "Electronics"}))
print(f"Electronics products: {len(electronics)}")

# Find products with price greater than $200
expensive_products = list(products.find({"price": {"$gt": 200}}))
print(f"Products over $200: {len(expensive_products)}")

# Find customers from a specific state
ca_customers = list(customers.find({"address.state": "CA"}))
print(f"Customers from California: {len(ca_customers)}")

# Find orders with specific status
completed_orders = list(orders.find({"status": "completed"}))
print(f"Completed orders: {len(completed_orders)}")

Electronics products: 6
Products over $200: 4
Customers from California: 3
Completed orders: 5


In [18]:
sample_customer = list(customers.find().limit(10))
for customer in sample_customer: 
    print(f"Customer_Name: (customer.get{'name', 'N/A'}") 

Customer_Name: John Doe
Customer_Name: Jane Smith
Customer_Name: Mike Johnson
Customer_Name: Sarah Wilson
Customer_Name: David Brown
Customer_Name: Lisa Davis
Customer_Name: Robert Garcia
Customer_Name: Emily Martinez
Customer_Name: James Anderson
Customer_Name: Maria Rodriguez


In [40]:
sample_reviews = list(products.find().limit(10))
for product in sample_reviews: 
    reviews = product.get('reviews', [])
    print("Reviews:")
    for review in reviews:
        print(review)
        print("---")

Reviews:
{'rating': 5, 'comment': 'Great sound quality!', 'user': 'john_doe', 'date': '2024-01-15'}
---
{'rating': 4, 'comment': 'Good value for money', 'user': 'jane_smith', 'date': '2024-01-20'}
---
{'rating': 5, 'comment': 'Excellent noise cancellation', 'user': 'audio_fan', 'date': '2024-02-01'}
---
Reviews:
{'rating': 5, 'comment': 'Perfect morning coffee!', 'user': 'coffee_lover', 'date': '2024-01-10'}
---
{'rating': 3, 'comment': 'Takes time to heat up', 'user': 'early_bird', 'date': '2024-01-25'}
---
{'rating': 4, 'comment': 'Good build quality', 'user': 'kitchen_guru', 'date': '2024-02-05'}
---
Reviews:
{'rating': 5, 'comment': 'Very comfortable!', 'user': 'runner_123', 'date': '2024-01-12'}
---
{'rating': 4, 'comment': 'Good for long runs', 'user': 'marathon_mike', 'date': '2024-01-18'}
---
{'rating': 5, 'comment': 'Great support', 'user': 'fitness_fan', 'date': '2024-02-02'}
---
Reviews:
{'rating': 5, 'comment': 'Amazing camera quality!', 'user': 'photo_pro', 'date': '2024-0

Question 2
 - 2.1. Write a query to find products with 5-star reviews using $elemMatch.
 - 2.2. Write a query to find products with reviews from a specific user (e.g., "john_doe") using $elemMatch.
 - 2.3. Write a query to find products with reviews that have both a rating of 4 or higher AND contain the word   "excellent" in the comment using $elemMatch.
 - 2.8. Explain the difference between querying arrays with and without $elemMatch.
 - querying arrays in mongodb without elemmatch allows each condition to match any element, so different conditions may be satisfied by different objects within the array. In contrast, using elemmatch requires all specified conditions to be met by the same array element, making the query more precise. 
 

In [ ]:
# Query nested fields using dot notation
ca_customers = list(customers.find({"address.state": "CA"}))
print(f"Customers from California: {len(ca_customers)}")

# Query arrays using $elemMatch
five_star_products = list(products.find({
    "reviews": {"$elemMatch": {"rating": 5}}
}))
print(f"Products with 5-star reviews: {len(five_star_products)}")

# Combine multiple conditions
high_priced_electronics = list(products.find({
    "$and": [
        {"category": "Electronics"},
        {"price": {"$gt": 100}}
    ]
}))

In [5]:
# 2.1: Find products where at least one review has a rating of 5
five_star_products = list(products.find({
    "reviews": {"$elemMatch": {"rating": 5}}
}))
print(f"Products with 5-star review: {len(five_star_products)}")


Products with 5-star review: 15


In [8]:
# Query arrays using $elemMatch
audio_fan_products = list(products.find({
    "reviews": {"$elemMatch": {"user": "audio_fan"}}
}))
print(f"products reviewed by Audio_fan: {len(audio_fan_products)}")


products reviewed by Audio_fan: 1


In [9]:
# Find products where at least one review has BOTH: rating >= 4 and containing the word "excellent" (case insensitive)
excellent_products = list(products.find({
    "reviews": {"$elemMatch": {
        "rating": {"$gte": 4},
        "comment": {"$regex": "excellent", "$options": "i"}
    }}
}))
print(f"Products with reviews having rating >= 4 and 'excellent' in comment: {len(excellent_products)}")

Products with reviews having rating >= 4 and 'excellent' in comment: 3


Question 3
 - 3.1. Write an aggregation to calculate average rating for each product.
 - 3.2. Write an aggregation to find the most expensive product in each category.
 - 3.3. Write an aggregation to count customers by state.
 - 3.4. Write an aggregation to find total revenue by product category from orders.
 - 3.5. Write an aggregation to find customers who have made more than 1 order.
 - 3.6. Explain how the $unwind operator works and why it is useful for array data.
The $unwind operator in mongodb seperates array fields so that each element becomes its own document, allowing for element level analysis. This enables calculations and aggregations i.e. counting, averageing, and joining with nested fields that would be impossible without $unwind. By using $unwind, you can easily group, sort, or match array elements for advanced analytics in aggregation pipelines. 

In [11]:
# Calculate average rating for each product using $unwind
pipeline = [
    {"$unwind": "$reviews"},  # Create one document per review
    {"$group": {
        "_id": "$product_id",
        "product_name": {"$first": "$name"},
        "avg_rating": {"$avg": "$reviews.rating"},
        "review_count": {"$sum": 1}
    }},
    {"$sort": {"avg_rating": -1}}
]

top_rated = list(products.aggregate(pipeline))
print("Top rated products:")
for product in top_rated[:3]:  # Show top 3
    print(f"{product['product_name']}: {product['avg_rating']:.2f}")

Top rated products:
Wireless Headphones: 4.67
Gaming Mouse: 4.67
Laptop: 4.67


In [13]:
# Count products by category with average price
category_pipeline = [
    {"$group": {
        "_id": "$category",
        "count": {"$sum": 1},
        "avg_price": {"$avg": "$price"},
        "total_stock": {"$sum": "$stock"}
    }},
    {"$sort": {"count": -1}}
]

category_stats = list(products.aggregate(category_pipeline))
print("Products by category:")
for category in category_stats:
    print(f"{category['_id']}: {category['count']} products, avg price: ${category['avg_price']:.2f}")

# Find customers by state with newsletter subscription rates
state_pipeline = [
    {"$group": {
        "_id": "$address.state",
        "customer_count": {"$sum": 1},
        "newsletter_subscribers": {
            "$sum": {"$cond": [{"$eq": ["$preferences.newsletter", True]}, 1, 0]}
        }
    }},
    {"$sort": {"customer_count": -1}}
]

state_stats = list(customers.aggregate(state_pipeline))
print("\nCustomers by state:")
for state in state_stats:
    print(f"{state['_id']}: {state['customer_count']} customers, {state['newsletter_subscribers']} newsletter subscribers")

Products by category:
Electronics: 6 products, avg price: $479.99
Sports: 5 products, avg price: $106.99
Kitchen: 4 products, avg price: $112.49

Customers by state:
TX: 4 customers, 3 newsletter subscribers
CA: 3 customers, 1 newsletter subscribers
WA: 1 customers, 1 newsletter subscribers
AZ: 1 customers, 0 newsletter subscribers
PA: 1 customers, 1 newsletter subscribers
FL: 1 customers, 1 newsletter subscribers
IL: 1 customers, 1 newsletter subscribers
OH: 1 customers, 0 newsletter subscribers
NC: 1 customers, 1 newsletter subscribers
NY: 1 customers, 1 newsletter subscribers


In [12]:
# Calculate total revenue by product category from orders using $lookup
revenue_pipeline = [
    {"$unwind": "$items"},  # Create one document per order item
    {"$lookup": {
        "from": "products",
        "localField": "items.product_id",
        "foreignField": "product_id",
        "as": "product_info"
    }},
    {"$unwind": "$product_info"},  # Unwind the joined product data
    {"$group": {
        "_id": "$product_info.category",
        "revenue": {"$sum": {"$multiply": ["$items.price", "$items.quantity"]}}
    }},
    {"$sort": {"revenue": -1}}
]

revenue_by_category = list(orders.aggregate(revenue_pipeline))
print("Revenue by category:")
for category in revenue_by_category:
    print(f"{category['_id']}: ${category['revenue']:.2f}")

Revenue by category:
Electronics: $2979.93
Sports: $1074.90
Kitchen: $819.92


In [15]:
# Aggregation pipeline to calculate average rating for each product

pipeline_avg_rating = [
    # Step 1: Unwind the reviews array so each review becomes its own document
    {"$unwind": "$reviews"},
    # Step 2: Group by the product's unique _id (change to "$product_id" if that's your PK)
    {"$group": {
        "_id": "$_id",                  # Group by product's unique identifier
        "product_name": {"$first": "$name"},  # Keep the product name
        "avg_rating": {"$avg": "$reviews.rating"},  # Calculate average of review ratings
        "review_count": {"$sum": 1}               # Count the number of reviews
    }},
    # Step 3: Sort the output documents by average rating in descending order
    {"$sort": {"avg_rating": -1}}
]

# Execute the aggregation pipeline and get results as a list
avg_ratings = list(products.aggregate(pipeline_avg_rating))

# Print each product's name and its average rating
for product in avg_ratings:
    print(f"{product['product_name']}: {product['avg_rating']:.2f} average rating")


Wireless Headphones: 4.67 average rating
Air Fryer: 4.67 average rating
Running Shoes: 4.67 average rating
Smart Watch: 4.67 average rating
Tennis Racket: 4.67 average rating
Microwave: 4.67 average rating
Gaming Mouse: 4.67 average rating
Laptop: 4.67 average rating
Smartphone: 4.67 average rating
Basketball: 4.67 average rating
Yoga Mat: 4.33 average rating
Blender: 4.33 average rating
Tablet: 4.33 average rating
Dumbbells Set: 4.33 average rating
Coffee Maker: 4.00 average rating


In [20]:
# aggregation pipeline to find the most expensive product in each category

most_expensive_pipeline = [ 
# sort products by price in descending order so the highest price appears first in each category 
    {"$sort": {"price": -1}}, 
# group by category, and for each group 
    # max_price use $first to capture the highest price (due to the previoius sort) 
    # product name use $first to capture the corresponding product's name 
    {"$group": { 
        "_id": "$category",  # group by the category field 
        "max_price": {"$first": "$price"}, # take the price from the first document for each category 
        "product_name": {"$first": "$name"} # take the name from the first document for each category 
    }} 
] 

# execute the aggregation pipeline and get results as a list  
most_expensive = list(products.aggregate(most_expensive_pipeline)) 

# print each category's most expensive product name and its prize 
for category in most_expensive: 
    print(f"{category['_id']}: {category['product_name']} at ${category['max_price']:.2f}")

Electronics: Laptop at $1299.99
Sports: Dumbbells Set at $199.99
Kitchen: Microwave at $159.99


In [24]:
# Aggregation pipeline to count customers in each state

pipeline_customers_state = [
    # Group documents by the 'address.state' field
    {"$group": {
        "_id": "$address.state",          # Group by state; each unique state becomes a _id in the result
        "customer_count": {"$sum": 1}     # Count the number of customers in each state by summing 1 for each document
    }},
    # Sort the grouped results by customer_count in descending order
    {"$sort": {"customer_count": -1}}
]

# Execute the aggregation pipeline and convert the result into a list
customers_by_state = list(customers.aggregate(pipeline_customers_state))

# Print each state along with how many customers it has
for state in customers_by_state:
    print(f"{state['_id']}: {state['customer_count']} customers")


TX: 4 customers
CA: 3 customers
WA: 1 customers
AZ: 1 customers
PA: 1 customers
FL: 1 customers
IL: 1 customers
OH: 1 customers
NC: 1 customers
NY: 1 customers


In [23]:
# Aggregation pipeline to calculate total revenue by product category from order data

pipeline_revenue_category = [
    # Unwind the items array so every order item becomes its own document for analysis
    {"$unwind": "$items"},  
    # Use $lookup to join the corresponding product info using the product_id from each item
    {"$lookup": {
        "from": "products",                 # Reference the products collection
        "localField": "items.product_id",   # Match items.product_id in orders
        "foreignField": "product_id",       # ...with product_id in products collection
        "as": "product_info"                # Output the matched product as an array in product_info
    }},
    # Unwind product_info array so each item is paired with its detailed product info
    {"$unwind": "$product_info"},
    # Group by product category and sum revenue for all items in that category
    {"$group": {
        "_id": "$product_info.category",         # Group the results by product category
        "total_revenue": {                       # Sum up revenue per group
            "$sum": {"$multiply": ["$items.price", "$items.quantity"]}  # Revenue for each item = price * quantity
        }
    }},
    # Sort categories by total revenue in descending order
    {"$sort": {"total_revenue": -1}}
]

# Execute the aggregation pipeline and get results as a list
revenue_by_category = list(orders.aggregate(pipeline_revenue_category))

# Print each category along with total revenue generated
for category in revenue_by_category:
    print(f"{category['_id']}: ${category['total_revenue']:.2f} revenue")

Electronics: $2979.93 revenue
Sports: $1074.90 revenue
Kitchen: $819.92 revenue


In [34]:
pipeline_multiple_orders = [
    # Group orders by customer.id if the customer field is nested
    {"$group": {
        "_id": "$customer.id",              # Use customer.id as the grouping key (adjust if needed)
        "order_count": {"$sum": 1}          # Count total orders per customer
    }},
    # Filter to keep only customers with more than one order
    {"$match": {"order_count": {"$gt": 1}}},
    # Sort in descending order by order_count
    {"$sort": {"order_count": -1}}
]

multiple_orders_customers = list(orders.aggregate(pipeline_multiple_orders))

for customer in multiple_orders_customers:
    print(f"Customer {customer['_id']}: {customer['order_count']} orders")

Customer None: 10 orders


Question 4
 - 4.1. Convert MongoDB data to pandas DataFrame and perform basic analysis.
 - 4.2. Analyze customer data by state and preferences.
 - 4.3. Calculate inventory value by category.
 - 4.4. Analyze order patterns by month.
 - 4.5. Find the most popular product categories based on order quantities.
 - 4.6. What are the benefits of using MongoDB with Python data analysis libraries?
combining mongodb with python's data analysis libraries lets you store and query complex, nested data efficiently, then perform advanced analytics, transformation, and visualization in pandas and other tools. This workflow allows seamless extration, cleaning, grouping, and analysis that would be difficult within mongodb alone. As a result, teams can make use of both powerful, flexible databases and rich, user friendly analytics environments for better real world insights. 

In [41]:
import pandas as pd
from pymongo import MongoClient

# Convert MongoDB collection to DataFrame
def mongo_to_dataframe(collection, query=None):
    if query is None:
        query = {}
    cursor = collection.find(query)
    return pd.DataFrame(list(cursor))

# Get products data
products_df = mongo_to_dataframe(products)
print(f"Products DataFrame shape: {products_df.shape}")

Products DataFrame shape: (15, 10)


In [36]:
# Analyze product data with pandas
products_df = mongo_to_dataframe(products)
print("Product Analysis:")
print(f"Total products: {len(products_df)}")
print(f"Average price: ${products_df['price'].mean():.2f}")
print(f"Price range: ${products_df['price'].min():.2f} - ${products_df['price'].max():.2f}")

# Category analysis
category_analysis = products_df.groupby('category').agg({
    'price': ['mean', 'count'],
    'stock': 'sum'
}).round(2)
print("\nCategory Analysis:")
print(category_analysis)

# Calculate inventory value
products_df['inventory_value'] = products_df['price'] * products_df['stock']
inventory_by_category = products_df.groupby('category')['inventory_value'].sum().sort_values(ascending=False)
print("\nInventory Value by Category:")
print(inventory_by_category)

Product Analysis:
Total products: 15
Average price: $257.66
Price range: $24.99 - $1299.99

Category Analysis:
              price       stock
               mean count   sum
category                       
Electronics  479.99     6   158
Kitchen      112.49     4    77
Sports       106.99     5   137

Inventory Value by Category:
category
Electronics    45898.42
Sports         10023.63
Kitchen         8199.23
Name: inventory_value, dtype: float64


In [37]:
# Work with customer data and nested addresses
customers_df = mongo_to_dataframe(customers)
print(f"Total customers: {len(customers_df)}")

# Extract state information from nested address
customers_df['state'] = customers_df['address'].apply(lambda x: x['state'])
state_analysis = customers_df['state'].value_counts()
print("\nCustomers by State:")
print(state_analysis)

# Analyze newsletter subscriptions
newsletter_subscribers = customers_df['preferences'].apply(lambda x: x['newsletter']).sum()
print(f"\nNewsletter subscribers: {newsletter_subscribers}")
print(f"Subscription rate: {newsletter_subscribers/len(customers_df)*100:.1f}%")

Total customers: 15

Customers by State:
state
TX    4
CA    3
NY    1
IL    1
AZ    1
PA    1
FL    1
OH    1
NC    1
WA    1
Name: count, dtype: int64

Newsletter subscribers: 10
Subscription rate: 66.7%


In [38]:
# Explode order items array into separate rows for analysis
orders_df = mongo_to_dataframe(orders)
items_df = pd.DataFrame([item for row in orders_df["items"] for item in row])
print(f"Total order items: {len(items_df)}")

# Join with product information to get categories
products_df = mongo_to_dataframe(products)
items_df = items_df.merge(
    products_df[["product_id", "category"]],
    on="product_id",
    how="left"
)

# Analyze items by category
category_analysis = items_df.groupby("category").agg({
    "quantity": "sum",
    "price": "mean"
}).round(2)
print("\nOrder items by category:")
print(category_analysis)

Total order items: 24

Order items by category:
             quantity   price
category                     
Electronics         7  425.70
Kitchen             8  102.49
Sports             10  116.10


In [42]:
orders_df = mongo_to_dataframe(orders)
# Convert order date to datetime (adjust 'order_date' if schema field name differs)
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
# Extract year and month for grouping
orders_df['year_month'] = orders_df['order_date'].dt.to_period('M')
orders_per_month = orders_df['year_month'].value_counts().sort_index()
print("Order count by month:")
print(orders_per_month)

Order count by month:
year_month
2024-01    3
2024-02    7
Freq: M, Name: count, dtype: int64


/tmp/ipykernel_310/1798057017.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  orders_df['year_month'] = orders_df['order_date'].dt.to_period('M')


In [43]:
# Flatten items arrays so each order item is a row
items_df = pd.DataFrame([item for items in orders_df['items'] for item in items])
# Merge with products to get category info
products_simple = products_df[['product_id', 'category']]
items_df = items_df.merge(products_simple, on='product_id', how='left')
# Sum ordered quantities by category
popular_categories = items_df.groupby('category')['quantity'].sum().sort_values(ascending=False)
print("Most popular product categories by quantity ordered:")
print(popular_categories)

Most popular product categories by quantity ordered:
category
Sports         10
Kitchen         8
Electronics     7
Name: quantity, dtype: int64


Questions 5
 - 5.1. Create appropriate indexes for the e-commerce collections.
 - 5.2. Test query performance using db.command("explain", …​) method.
 - 5.3. Create compound and text indexes for advanced queries.
 - 5.4. Create indexes on nested fields like reviews.rating.
 - 5.5. Test the performance of complex aggregation pipelines.
 - 5.6. What are the key considerations for optimizing MongoDB performance in production?

In [45]:
# Create indexes for better performance
products.create_index("category")
products.create_index("price")
products.create_index([("category", 1), ("price", -1)])  # Compound index

# Test query performance
query = {"category": "Electronics"}
explain_result = db.command("explain", {"find": "products", "filter": query})
stats = explain_result.get("executionStats", {})
print("\nQuery Performance (category = Electronics):")
print("Execution time (ms):", stats.get("executionTimeMillis", "N/A"))
print("Documents examined:", stats.get("totalDocsExamined", "N/A"))
print("Documents returned:", stats.get("nReturned", stats.get("totalDocsReturned", "N/A")))


Query Performance (category = Electronics):
Execution time (ms): 1
Documents examined: 6
Documents returned: 6


In [46]:
# Create additional specialized indexes
products.create_index("tags")  # Multikey index for arrays
products.create_index("specifications.brand")  # Index on nested field
customers.create_index("email", unique=True)  # Unique index
orders.create_index([("customer_id", 1), ("order_date", -1)])  # Compound index
products.create_index([("name", "text")])  # Text index for product names
print("\nCreated specialized indexes (tags, brand, email(unique), compound, text).")

# Test different query patterns
queries_to_test = [
    {"category": "Electronics"},
    {"tags": "wireless"},
    {"specifications.brand": "Samsung"},
    {"price": {"$gt": 200, "$lt": 500}},
]

print("\nQuery Performance Analysis:")
for i, query in enumerate(queries_to_test, 1):
    result = db.command("explain", {"find": "products", "filter": query})
    stats = result.get("executionStats", {})
    print(f"Query {i}: {query}")
    print("  Execution time (ms):", stats.get("executionTimeMillis", "N/A"))
    print("  Docs examined:", stats.get("totalDocsExamined", "N/A"))
    print("  Docs returned:", stats.get("nReturned", stats.get("totalDocsReturned", "N/A")))
    print()


Created specialized indexes (tags, brand, email(unique), compound, text).

Query Performance Analysis:
Query 1: {'category': 'Electronics'}
  Execution time (ms): 0
  Docs examined: 6
  Docs returned: 6

Query 2: {'tags': 'wireless'}
  Execution time (ms): 0
  Docs examined: 1
  Docs returned: 1

Query 3: {'specifications.brand': 'Samsung'}
  Execution time (ms): 0
  Docs examined: 0
  Docs returned: 0

Query 4: {'price': {'$gt': 200, '$lt': 500}}
  Execution time (ms): 0
  Docs examined: 2
  Docs returned: 2



In [47]:
# Check index usage statistics
db_stats = db.command("collStats", "products")
print("Collection Statistics:")
print(f"Total documents: {db_stats.get('count', 'N/A')}")
print(f"Average document size: {db_stats.get('avgObjSize', 0):.2f} bytes")
print(f"Total collection size: {db_stats.get('size', 0)/1024:.2f} KB")
print(f"Total index size: {db_stats.get('totalIndexSize', 0)/1024:.2f} KB")

# List all indexes
indexes = list(products.list_indexes())
print(f"\nIndexes on 'products' collection: {len(indexes)}")
for idx in indexes:
    print(f"- {idx['name']}: {idx['key']}")

# Test aggregation performance
agg_pipeline = [
    {"$group": {"_id": "$category", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
agg_explain = db.command("explain", {"aggregate": "products", "pipeline": agg_pipeline, "cursor": {}}, verbosity="executionStats")
print(f"\nAggregation execution time: {agg_explain.get('executionStats', {}).get('executionTimeMillis', 'N/A')}ms")

Collection Statistics:
Total documents: 15
Average document size: 714.00 bytes
Total collection size: 10.46 KB
Total index size: 140.00 KB

Indexes on 'products' collection: 7
- _id_: SON([('_id', 1)])
- category_1: SON([('category', 1)])
- price_1: SON([('price', 1)])
- category_1_price_-1: SON([('category', 1), ('price', -1)])
- tags_1: SON([('tags', 1)])
- specifications.brand_1: SON([('specifications.brand', 1)])
- name_text: SON([('_fts', 'text'), ('_ftsx', 1)])

Aggregation execution time: N/Ams
